# Linear Regression Assignment

This notebook implements Linear Regression in two ways:

1. Using scikit-learn `LinearRegression`
2. From scratch using batch gradient descent

Finally, it compares coefficients, intercept, and error values.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Allow importing files from the assignment root if notebook is opened from notebooks folder.
assignment_root = Path.cwd()
if assignment_root.name == 'notebooks':
    assignment_root = assignment_root.parent
sys.path.append(str(assignment_root))


## 1. Load Processed Dataset


In [ ]:
dataset_path = assignment_root / 'data' / 'linear_regression_dataset.csv'
df = pd.read_csv(dataset_path)
df.head()


In [ ]:
FEATURE_COLUMNS = ['study_hours', 'attendance_percent', 'previous_score']
TARGET_COLUMN = 'final_score'

X = df[FEATURE_COLUMNS].to_numpy(dtype=float)
y = df[TARGET_COLUMN].to_numpy(dtype=float)

print('Feature shape:', X.shape)
print('Target shape:', y.shape)


## 2. Linear Regression using scikit-learn


In [ ]:
sklearn_model = LinearRegression()
sklearn_model.fit(X, y)

sklearn_predictions = sklearn_model.predict(X)
sklearn_mse = mean_squared_error(y, sklearn_predictions)
sklearn_r2 = r2_score(y, sklearn_predictions)

print('Scikit-learn coefficients:', sklearn_model.coef_)
print('Scikit-learn intercept:', sklearn_model.intercept_)
print('Scikit-learn MSE:', sklearn_mse)
print('Scikit-learn R2:', sklearn_r2)


## 3. Linear Regression from Scratch using Gradient Descent

Formula used:

- Prediction: `y_pred = Xw + b`
- Loss: `MSE = mean((y_pred - y_actual)^2)`
- Weight update: `w = w - learning_rate * dw`
- Bias update: `b = b - learning_rate * db`


In [ ]:
class LinearRegressionFromScratch:
    def __init__(self, learning_rate=0.05, epochs=20000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.feature_means = None
        self.feature_stds = None
        self.coefficients_ = None
        self.intercept_ = None
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        self.feature_means = X.mean(axis=0)
        self.feature_stds = X.std(axis=0)
        self.feature_stds[self.feature_stds == 0] = 1
        X_scaled = (X - self.feature_means) / self.feature_stds

        n_samples, n_features = X_scaled.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.epochs):
            predictions = np.dot(X_scaled, self.weights) + self.bias
            errors = predictions - y

            dw = (2 / n_samples) * np.dot(X_scaled.T, errors)
            db = (2 / n_samples) * np.sum(errors)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            self.loss_history.append(np.mean(errors ** 2))

        # Convert scaled coefficients back to original feature scale.
        self.coefficients_ = self.weights / self.feature_stds
        self.intercept_ = self.bias - np.sum((self.weights * self.feature_means) / self.feature_stds)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.dot(X, self.coefficients_) + self.intercept_


In [ ]:
custom_model = LinearRegressionFromScratch(learning_rate=0.05, epochs=20000)
custom_model.fit(X, y)
custom_predictions = custom_model.predict(X)
custom_mse = np.mean((custom_predictions - y) ** 2)

print('Custom coefficients:', custom_model.coefficients_)
print('Custom intercept:', custom_model.intercept_)
print('Custom MSE:', custom_mse)
print('Final training loss:', custom_model.loss_history[-1])


## 4. Compare Coefficients and Intercept


In [ ]:
comparison_df = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Scikit_Learn_Coefficient': sklearn_model.coef_,
    'Custom_GD_Coefficient': custom_model.coefficients_,
    'Absolute_Difference': np.abs(sklearn_model.coef_ - custom_model.coefficients_)
})
comparison_df


In [ ]:
intercept_comparison = pd.DataFrame({
    'Metric': ['Intercept', 'Mean Squared Error'],
    'Scikit_Learn': [sklearn_model.intercept_, sklearn_mse],
    'Custom_GD': [custom_model.intercept_, custom_mse],
    'Absolute_Difference': [
        abs(sklearn_model.intercept_ - custom_model.intercept_),
        abs(sklearn_mse - custom_mse)
    ]
})
intercept_comparison


## Conclusion

The custom gradient descent implementation learns coefficients and intercept very close to scikit-learn LinearRegression. Minor differences can happen due to numerical precision and gradient descent convergence settings.
